## Step 1: Creating Tokens

In [3]:
with open("E:\python\LLM-From-Scratch\dataset\Book7.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of text: ", len(raw_text))
print(raw_text[:50])

Total number of text:  1227024
I 



* 

THE DARK LORD ASCENDING 

The two men ap


In [ ]:
import re

In [5]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(preprocessed[:30])

['I', '*', 'THE', 'DARK', 'LORD', 'ASCENDING', 'The', 'two', 'men', 'appeared', 'out', 'of', 'nowhere', ',', 'a', 'few', 'yards', 'apart', 'in', 'the', 'narrow', ',', 'moonlit', 'lane', '.', 'For', 'a', 'second', 'they', 'stood']


In [6]:
print(len(preprocessed))


254161


## Step 2: Creating Token IDs

In [12]:
all_words = sorted(set(preprocessed))
print("Vocab SizeL: ", len(all_words))

Vocab SizeL:  14808


In [13]:
vocab = {token:integer for integer,token in enumerate(all_words)}


In [14]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
('*', 5)
(',', 6)
('-', 7)
('-J', 8)
('-Know-', 9)
('-□', 10)
('.', 11)
('/', 12)
('1', 13)
('10', 14)
('100', 15)
('101', 16)
('102', 17)
('103', 18)
('104', 19)
('105', 20)
('106', 21)
('107', 22)
('108', 23)
('109', 24)
('11', 25)
('110', 26)
('111', 27)
('112', 28)
('113', 29)
('114', 30)
('115', 31)
('116', 32)
('117', 33)
('118', 34)
('119', 35)
('12', 36)
('120', 37)
('121', 38)
('122', 39)
('123', 40)
('124', 41)
('125', 42)
('126', 43)
('127', 44)
('128', 45)
('129', 46)
('13', 47)
('130', 48)
('131', 49)
('132', 50)


In [15]:
class SimpleTokenizerv1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    
    def encode(self, text):

        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids


    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text



## Adding Special Tokens


In [23]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [24]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


### BYTE PAIR ENCODING (BPE)


In [25]:
! pip install tiktoken

Defaulting to user installation because normal site-packages is not writeable


In [26]:
import tiktoken


tokenizer = tiktoken.get_encoding("gpt2")

In [27]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [28]:
# Initialize the encodings for GPT-2, GPT-3, and GPT-4
encodings = {
    "gpt2": tiktoken.get_encoding("gpt2"),
    "gpt3": tiktoken.get_encoding("p50k_base"),  # Commonly associated with GPT-3 models
    "gpt4": tiktoken.get_encoding("cl100k_base")  # Used for GPT-4 and later versions
}

# Get the vocabulary size for each encoding
vocab_sizes = {model: encoding.n_vocab for model, encoding in encodings.items()}

# Print the vocabulary sizes
for model, size in vocab_sizes.items():
    print(f"The vocabulary size for {model.upper()} is: {size}")

The vocabulary size for GPT2 is: 50257
The vocabulary size for GPT3 is: 50281
The vocabulary size for GPT4 is: 100277


### CREATING INPUT-TARGET PAIRS

In [30]:
with open("E:\python\LLM-From-Scratch\dataset\Book7.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

379362


In [31]:
enc_sample = enc_text[50:]


In [32]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
#to predict the next word in the sequence. 
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [11, 266, 1746, 7924]
y:      [266, 1746, 7924, 379]


In [33]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[11] ----> 266
[11, 266] ----> 1746
[11, 266, 1746] ----> 7924
[11, 266, 1746, 7924] ----> 379


In [34]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

, ---->  w
, w ----> ands
, wands ---->  directed
, wands directed ---->  at


### IMPLEMENTING A DATA LOADER

In [35]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [36]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [38]:
import torch

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   220,   628,   198],
        [  198,     9,   220,   198],
        [  198, 10970, 46455, 35750],
        [25400, 10619,  2751,   220],
        [  198,   198,   464,   734],
        [ 1450,  4120,   503,   286],
        [12062,    11,   257,  1178],
        [ 5695,   220,   198,   499]])

Targets:
 tensor([[  220,   628,   198,   198],
        [    9,   220,   198,   198],
        [10970, 46455, 35750, 25400],
        [10619,  2751,   220,   198],
        [  198,   464,   734,  1450],
        [ 4120,   503,   286, 12062],
        [   11,   257,  1178,  5695],
        [  220,   198,   499,   433]])


### CREATING TOKEN EMBEDDINGS

In [39]:
input_ids = torch.tensor([2, 3, 5, 1])


In [40]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


### POSITIONAL EMBEDDINGS (ENCODING WORD POSITIONS)

In [42]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [43]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [44]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   220,   628,   198],
        [  198,     9,   220,   198],
        [  198, 10970, 46455, 35750],
        [25400, 10619,  2751,   220],
        [  198,   198,   464,   734],
        [ 1450,  4120,   503,   286],
        [12062,    11,   257,  1178],
        [ 5695,   220,   198,   499]])

Inputs shape:
 torch.Size([8, 4])


In [45]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


# Complete Pre-Processing Pipeline 

### Importing Libraries

In [47]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import tiktoken

### Creating Tokenizer (GPT-2 BPE)

In [48]:
class GPT2TokenizerWrapper:
    def __init__(self):
        self.enc = tiktoken.get_encoding("gpt2")

        # Special Tokens
        self.eos_token = self.enc.eot_token
        self.vocab_size = self.enc.n_vocab


    def encode(self, text):
        tokens = self.enc.encode(text)
        tokens.append(self.eos_token)
        return tokens
    
    
    def decode(self, tokens):
        return self.enc.decode(tokens)
    
    

### Create Input-Target Pairs

In [49]:

class TextDataset(Dataset):
    def __init__(self, text, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size

        tokens = tokenizer.encode(text)

        self.data = []
        for i in range(len(tokens) - block_size):
            x = tokens[i:i+block_size]
            y = tokens[i+1:i+block_size+1]
            self.data.append((x, y))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

### Creating Dataloader 

In [51]:
def get_dataloader(dataset, batch_size=4):
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


### Positional Embedding (Learned)

In [52]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        positions = torch.arange(0, seq_len).unsqueeze(0).repeat(batch_size, 1)
        return self.pos_emb(positions)

### Input Embedding (Token + Position)

In [53]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEmbedding(max_len, d_model)

    def forward(self, x):
        token = self.token_emb(x)
        pos = self.pos_emb(token)
        return token + pos

### FULL PIPELINE WRAPPER

In [54]:
class PreprocessingPipeline:
    def __init__(self, text, block_size=8, batch_size=2, d_model=32):

        # Step 1–3: Tokenizer
        self.tokenizer = GPT2TokenizerWrapper()

        # Step 4: Dataset
        self.dataset = TextDataset(text, self.tokenizer, block_size)

        # Step 5: Dataloader
        self.loader = get_dataloader(self.dataset, batch_size)

        # Step 6–7: Embedding
        self.embedding = InputEmbedding(
            vocab_size=self.tokenizer.vocab_size,
            d_model=d_model,
            max_len=block_size
        )

    def run(self):
        for x, y in self.loader:

            # Input IDs → Embeddings
            emb = self.embedding(x)

            print("Input IDs:\n", x)
            print("\nTarget IDs:\n", y)
            print("\nEmbedding Shape:", emb.shape)

            return emb

In [55]:
def load_text_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [56]:
if __name__ == "__main__":

    file_path = r"E:\python\LLM-From-Scratch\dataset\Book7.txt"

    text = load_text_file(file_path)

    print("Loaded text length:", len(text))

    pipeline = PreprocessingPipeline(
        text=text,
        block_size=64,   # increase for real data
        batch_size=8,
        d_model=64
    )

    output = pipeline.run()

Loaded text length: 1227024
Input IDs:
 tensor([[  198,   198,   447,   250,   564,   246, 42204,    11,   447,   247,
           326,   447,   247,    82,  5741,    11,   447,   251,  6575,  4893,
            13,   564,   250,  2990,   447,   247,   303,   477,  1392,   220,
           198,  8189,  3891,    11,   475,   345,   460,  3221,  1560,   851,
           564,   251,   220,   198,   198,   447,   250,  2484,    71,     0,
           447,   251,   531, 19959,    13,   220,   198,   198,   447,   250,
          1537,   878,   356,  3285],
        [30079,  5741,    13,   564,   250, 23061,   364,    11,   326,  6774,
           514,   284,   262,   886,   286,   220,   198, 29214, 14179,  8340,
            13,   775,   836,   447,   247,    83,   760,   618,   340,   481,
           307,   220,   198,    79,  4733,   284,  7025,   757,    11,   475,
           345,   460,   307,  1654,   356,   220,   198, 49271,   307,   736,
            13,  9175,   665, 41367,   883,  5980,   